In [3]:
import os
from dotenv import load_dotenv
#
load_dotenv()

#
# 1. Configure authentication credentials
# If running inside a Databricks Notebook, the host and token are automatically discovered.
DATABRICKS_HOST = os.environ.get("DATABRICKS_HOST")
DATABRICKS_TOKEN = os.environ.get("DATABRICKS_PERSONAL_ACCESS_TOKEN")

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks_langchain import ChatDatabricks, DatabricksMCPServer, DatabricksMultiServerMCPClient
from langchain.agents import create_agent
from com.example.ai.LLMManager import LLMManager
 
workspace_client = WorkspaceClient()
host = workspace_client.config.host

mcp_client = DatabricksMultiServerMCPClient([
    DatabricksMCPServer(
        name="uc-functions",
        url=f"{host}/api/2.0/mcp/functions/workspace/default",
        workspace_client=workspace_client,
    ),
])

async with mcp_client:
    tools = await mcp_client.get_tools()
    agent = create_agent(
        ChatDatabricks(endpoint="databricks-gemma-3-12b"),
        tools=tools,
    )
    result = await agent.ainvoke(
        {"messages": [{"role": "user", "content": "Look up customer info for Acme Corp"}]}
    )
    print(result["messages"][-1].content)



In [9]:
import asyncio
import nest_asyncio
from databricks.sdk import WorkspaceClient
from databricks_langchain import ChatDatabricks, DatabricksMCPServer, DatabricksMultiServerMCPClient
from langchain.agents import create_agent
from com.example.ai.LLMManager import LLMManager

nest_asyncio.apply()
 
workspace_client = WorkspaceClient()
host = workspace_client.config.host

databricks_mcp_client = DatabricksMultiServerMCPClient(
    [
        DatabricksMCPServer(
            name="system-ai",
            url=f"{host}/api/2.0/mcp/functions/system/ai",
        ),
        DatabricksMCPServer(
            name="custom_mcp",
            url=f"{host}/api/2.0/mcp/sql",
        #     workspace_client=custom_mcp_server_workspace_client
        ),
        # DatabricksMCPServer(
        #     name="obo_vs_client",
        #     url=f"{host}/api/2.0/mcp/vector-search/system/ai",
        #     workspace_client=obo_workspace_client
        # )
    ]
)

# Create MCP tools from the configured servers
mcp_tools = asyncio.run(databricks_mcp_client.get_tools())

agent = create_agent(
    ChatDatabricks(endpoint="databricks-gemma-3-12b"),
    tools=mcp_tools,
)
# Create the agent graph with an LLM, tool set, and system prompt (if given)
#agent = create_tool_calling_agent(llm, mcp_tools, system_prompt)

result = await agent.ainvoke(
        {"messages": [{"role": "user", "content": "List all unity catalogs for workspace."}]}
    )
print(result["messages"][-1].content)

The available unity catalogs are samples, system, and workspace.


In [10]:
result = await agent.ainvoke(
        {"messages": [{"role": "user", "content": "Show all squares for 25 to 30"}]}
    )
print(result["messages"][-1].content)

/home/brijeshdhaker/snap/code/243/.local/share/uv/python/cpython-3.13.13-linux-x86_64-gnu/lib/python3.13/logging/__init__.py:789: RuntimeWarning: coroutine 'DatabricksMultiServerMCPClient.get_tools' was never awaited
  def __init__(self, name=''):


The squares of the numbers from 25 to 30 are: 625, 676, 729, 784, 841, and 900.


In [11]:
result = await agent.ainvoke(
        {"messages": [{"role": "user", "content": "SHOW TABLES IN samples.accuweather"}]}
    )
print(result["messages"][-1].content)

The following tables are available in the `samples.accuweather` catalog: forecast_daily_calendar_imperial, forecast_daily_calendar_metric, forecast_daynight_imperial, forecast_daynight_metric, forecast_hourly_imperial, forecast_hourly_metric, historical_daily_calendar_imperial, historical_daily_calendar_metric, historical_daynight_imperial, historical_daynight_metric, historical_hourly_imperial, and historical_hourly_metric.


In [12]:
result = await agent.ainvoke(
        {"messages": [{"role": "user", "content": "DESCRIBE TABLE samples.nyctaxi.trips"}]}
    )
print(result["messages"][-1].content)

The table `samples.nyctaxi.trips` has 6 columns: `tpep_pickup_datetime` (timestamp), `tpep_dropoff_datetime` (timestamp), `trip_distance` (double), `fare_amount` (double), `pickup_zip` (int), and `dropoff_zip` (int).
